<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/codex/MLR-20-eprime-wandb-integration/notebooks/demo_spotlight_e_prime_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spotlight E-Prime Constraint Retention In Colab

This notebook runs a long-conversation E-Prime constraint-retention experiment with baseline generation and Spotlight-steered generation. It uses the CUDA/vLLM Colab `HookLLM` path and intentionally avoids Apple Silicon/Metal support.


### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo and installs the CUDA-compatible notebook dependencies.


In [ ]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
#
# PURPOSE:
# This cell prepares the environment to run vLLM and its custom plugins.
# It handles complex dependency conflicts common in Colab (e.g., pre-installed
# torch versions) and ensures the plugin code is loaded correctly.
#
# KEY ACTIONS:
# 1. Clones the vLLM-Hook repository if not present.
# 2. Installs a compatible version of vLLM and PyTorch for your GPU.
# 3. Cleans up old binary artifacts to prevent version conflicts.
# 4. Loads the plugin source code.
# 5. TRIGGERS A COLAB RESTART: The cell will restart the runtime to ensure
#    the new libraries are fully loaded into the kernel memory.
#
# EXPECTED BEHAVIOR:
# - The cell will run and install packages.
# - It may print "Restarting Colab runtime...".
# - The cell will stop abruptly, and the runtime will restart.
# - The restart is automatic; the code will resume from the top.
# ==============================================================================

from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys
import time

# Configuration
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "codex/MLR-20-eprime-wandb-integration")
REPO_NAME = "vLLM-Hook"

# Environment Variables (Optional overrides)
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()

# --------------------------------------------------------------------------
# Helper Functions
# --------------------------------------------------------------------------

def run(cmd, cwd=None, env=None):
    """
    Executes a shell command, printing output in real-time.
    Raises an error if the command fails.
    """
    cmd = [str(part) for part in cmd]
    print(f"> Running: {' '.join(cmd)}", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = tail[-120:]

    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )

def run_capture(cmd):
    """Executes a command and returns stdout/stderr without printing."""
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)

def norm(name):
    """Normalize package names for comparison."""
    return name.lower().replace("_", "-")

def package_from_req_line(line: str) -> str:
    """Extract package name from a requirement string (e.g., 'torch>=1.0' -> 'torch')."""
    stripped = line.strip()
    # Split on version specifiers
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())

def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Checks if the git repo's origin matches the expected URL."""
    try:
        url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return url == expected_remote

def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Searches up the directory tree for a matching git repo."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None

def assert_cuda_runtime():
    """Ensures a GPU is available before proceeding."""
    try:
        import torch
    except Exception:
        torch = None

    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None

    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "No CUDA GPU detected. "
            "In Colab, go to Runtime > Change runtime type and select T4 GPU (or better), "
            "then re-run the entire notebook from the beginning."
        )

# --------------------------------------------------------------------------
# 1. Repository Setup
# --------------------------------------------------------------------------

expected_remote = REPO_URL.removesuffix(".git")
existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)

if IN_COLAB:
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"✓ Reusing existing repo at: {REPO_ROOT}")
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch detected. Replacing clone...")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Refreshing existing clone...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Working directory set to: {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print(f"\n--- Environment Summary ---")
print(f"Running in Colab: {IN_COLAB}")
print(f"Repo Root: {REPO_ROOT}")
print(f"Plugin Dir: {PKG_DIR}")

if IN_COLAB:
    assert_cuda_runtime()

# --------------------------------------------------------------------------
# 2. Plugin Directory Validation
# --------------------------------------------------------------------------

if not PKG_DIR.exists():
    raise FileNotFoundError(
        f"Plugin directory not found at {PKG_DIR}. "
        "Please ensure the repository was cloned correctly."
    )

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("git is required but unavailable in this runtime.")

# --------------------------------------------------------------------------
# 3. Dependency Management
# --------------------------------------------------------------------------

if REQ_FILE.exists():
    keep = []
    # These packages are managed by Colab's pre-installed environment.
    # Installing them again can cause conflicts.
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}

    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue

        package = package_from_req_line(stripped)
        if package in blocked:
            print(f"⏭ Skipping managed dependency: {line}")
            continue
        keep.append(line)

    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    print(f"Installing filtered requirements from {REQ_FILE.name}...")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("⚠ Warning: No requirement.txt found; skipping custom dependency installation.")

# Ensure protobuf is at the correct version
print("Ensuring protobuf version compatibility...")
run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

# --------------------------------------------------------------------------
# 4. vLLM & PyTorch Installation
# --------------------------------------------------------------------------

if COLAB_INSTALL_VLLM:
    print(f"Installing user-specified vLLM: {COLAB_INSTALL_VLLM}")
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    print(f"\nInstalling vLLM and PyTorch for {VLLM_TORCH_BACKEND}...")

    # 1. Uninstall existing conflicting versions
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])

    # 2. Manually remove leftover binary artifacts that pip might miss
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        leftovers = [
            site_path / "vllm", *site_path.glob("vllm-*.dist-info"),
            site_path / "torch", *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision", *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio", *site_path.glob("torchaudio-*.dist-info"),
        ]
        for leftover in leftovers:
            if leftover.exists():
                print(f"  Cleaning up leftover: {leftover.name}")
                if leftover.is_dir():
                    shutil.rmtree(leftover)
                else:
                    leftover.unlink()

    # 3. Force clear GPU memory
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    # 4. Install using 'uv' (faster than pip) with specific CUDA backend
    print("  Downloading and installing packages with 'uv'...")
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

    # 5. Verify Scipy/Numpy
    scipy_check = run_capture([
        sys.executable, "-c", "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)"
    ])
    if scipy_check.returncode:
        print("  Detected numpy/scipy issues; upgrading...")
        run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

# Verify Installation
print("\nVerifying installation...")
run([
    sys.executable,
    "-c",
    "import torch, vllm; print('✓ torch:', torch.__version__); print('✓ vllm:', getattr(vllm, '__version__', 'unknown'))"
])

# --------------------------------------------------------------------------
# 5. Plugin Loading
# --------------------------------------------------------------------------

# Strategy: Add path to sys.path to allow immediate import,
# then ensure metadata is installed for subprocesses later.
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

try:
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("✓ Plugin module loaded successfully from source path.")
except Exception as e:
    print(f"⚠ Warning: Initial import check failed ({e}). This is expected if compilation is needed later.")

# Ensure the package is registered in the environment (metadata)
# This is necessary for tools that rely on `importlib.metadata`.
print("Registering plugin package in environment...")
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])

print("Installing W&B tracking client...")
run([sys.executable, "-m", "pip", "install", "wandb"])

print(f"Plugin Source: {plugin_src_dir}")
print(f"Python Exec  : {sys.executable}")

# --------------------------------------------------------------------------
# 6. Runtime Restart (Critical for Colab)
# --------------------------------------------------------------------------
# We restart the runtime to ensure the newly installed binary libraries
# are loaded into a fresh Python interpreter. This prevents "dirty state" issues.
if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("\n" + "="*50)
    print("RESTARTING COLAB RUNTIME...")
    print("This ensures the new vLLM/Torch binaries are fully loaded.")
    print("Do not interrupt this process.")
    print("="*50)

    time.sleep(1) # Allow output buffer to flush
    sys.exit(0) # Terminate this process to trigger Colab restart


### Scoring Dependencies

The E-Prime checker uses spaCy for state-of-being verb detection. Run this cell before scoring.


In [ ]:
# Optional scoring dependency setup for Colab.
# Run after the main setup cell. If you already have spaCy and en_core_web_sm, this is a no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])

try:
    import spacy
    spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


### Imports & Environment


In [ ]:
import io
import json
import os
import multiprocessing as mp
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins
from research.experiment_tracking import (
    ExperimentCondition,
    ExperimentResult,
    RunMetrics,
    TurnResult,
    WandbTracker,
    collect_runtime_provenance,
)

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


### Initialize `HookLLM`


In [ ]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 8192

llm = HookLLM(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=0.65,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("Spotlight worker enabled")
print(f"Max model length: {MAX_MODEL_LEN}")


### E-Prime Long-Conversation Configuration


In [ ]:
E_PRIME_CONSTRAINT = """Persistent E-Prime constraint:
Do not use state-of-being verbs: am, is, are, was, were, be, being, been.
Do not use contractions containing state-of-being verbs, including I'm, you're, we're, they're, he's, she's, it's, that's, and there's.
""".strip()

SYSTEM_MESSAGE = (
    "You are a concise assistant continuing a long Talk2AI-style conversation. "
    "Preserve the E-Prime constraint throughout the whole conversation while still answering helpfully."
)

SEED_HISTORY = [
    {"role": "user", "content": "I keep hearing that climate change discussions are exaggerated compared with other problems."},
    {"role": "assistant", "content": "Let's compare long-run measurements with the way headlines describe them."},
    {"role": "user", "content": "The headlines make me distrust the whole topic."},
    {"role": "assistant", "content": "Repeated alarm can make even strong evidence feel performative."},
]

USER_TURNS = [
    "What evidence should I look at if I want to avoid headline-driven conclusions?",
    "How do local weather experiences confuse the broader trend?",
    "What tradeoffs matter for households when policies raise energy costs?",
    "Where does adaptation make sense, and where does it fall short?",
    "How should poorer countries think about growth and emissions limits?",
    "What practical local policy question should I ask a city council candidate?",
    "How can I separate serious policy criticism from misinformation?",
    "What should climate communicators stop doing if they want skeptical people to listen?",
    "What should good-faith skeptics concede before debating policy?",
    "End with a cautious, practical position that still respects the evidence.",
]

ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=160)
MAX_USER_TURNS = min(10, len(USER_TURNS))
HISTORY_WINDOW_MESSAGES = 16


### Prompt And Run Helpers


In [ ]:
def render_e_prime_prompt(history, user_message):
    recent_history = history[-HISTORY_WINDOW_MESSAGES:]
    omitted_messages = max(0, len(history) - len(recent_history))
    transcript = "\n".join(
        f"{item['role'].upper()}: {item['content']}" for item in recent_history
    )
    if transcript:
        transcript += "\n"

    earlier_context = ""
    if omitted_messages:
        earlier_context = (
            f"Earlier conversation context: {omitted_messages} older messages are omitted "
            "from this prompt to keep the Colab run within memory limits. Continue the same conversation.\n\n"
        )

    return f"""{SYSTEM_MESSAGE}

{E_PRIME_CONSTRAINT}

{earlier_context}Recent conversation:
{transcript}USER: {user_message}
ASSISTANT:""".strip()


def clean_text(text):
    return (text or "").strip().split("\n\n")[0].strip()


def run_condition(name, use_spotlight):
    history = list(SEED_HISTORY)
    rows = []
    for turn_index, user_message in enumerate(USER_TURNS[:MAX_USER_TURNS], start=1):
        prompt = render_e_prime_prompt(history, user_message)
        started = time.perf_counter()
        if use_spotlight:
            # Hook-enabled path: generate_with_spotlight calls llm.generate(..., use_hook=True)
            # after converting the E-Prime constraint span into Spotlight token ranges.
            outputs = generate_with_spotlight(
                llm,
                prompts=[prompt],
                emph_strings=[E_PRIME_CONSTRAINT],
                alpha=ALPHA,
                sampling_params=SAMPLING_PARAMS,
            )
        else:
            outputs = llm.generate(
                prompts=[prompt],
                sampling_params=SAMPLING_PARAMS,
                use_hook=False,
            )
        latency_s = time.perf_counter() - started
        reply = clean_text(outputs[0].outputs[0].text)
        rows.append(
            {
                "condition": name,
                "turn": turn_index,
                "prompt_chars": len(prompt),
                "history_messages_in_prompt": min(len(history), HISTORY_WINDOW_MESSAGES),
                "latency_s": latency_s,
                "user_message": user_message,
                "reply": reply,
            }
        )
        history.extend([
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": reply},
        ])
        print(f"{name} turn {turn_index}: prompt_chars={len(prompt)} latency={latency_s:.2f}s")
    return rows


### Run Baseline And Spotlight Conditions


In [ ]:
baseline_rows = run_condition("baseline", use_spotlight=False)
spotlight_rows = run_condition("spotlight", use_spotlight=True)
results = pd.DataFrame(baseline_rows + spotlight_rows)
results


### Score E-Prime Constraint Retention


In [ ]:
STATE_OF_BEING_CORE_WORDS = {"am", "is", "are", "was", "were", "be", "being", "been"}
STATE_OF_BEING_SPACY_MODEL = "en_core_web_sm"
CONTRACTION_PATTERN = re.compile(
    r"\b(?:i'm|you're|we're|they're|he's|she's|it's|that's|there's|what's|who's|where's|when's|why's|how's)\b",
    re.IGNORECASE,
)


def load_state_of_being_nlp(model_name=STATE_OF_BEING_SPACY_MODEL):
    try:
        import spacy
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "spaCy is required for the E-Prime checker. Run the scoring dependency setup cell first."
        ) from exc

    try:
        return spacy.load(model_name, disable=["ner", "parser"])
    except OSError as exc:
        raise OSError(
            f"spaCy model {model_name!r} is not installed. Run the scoring dependency setup cell first."
        ) from exc


state_of_being_nlp = load_state_of_being_nlp()


def is_state_of_being_token(token):
    lemma = token.lemma_.lower()
    text = token.text.lower()
    if lemma == "be" and token.pos_ in {"AUX", "VERB"}:
        return True
    return text in STATE_OF_BEING_CORE_WORDS and token.pos_ in {"AUX", "VERB"}


def check_e_prime_violations(text, nlp=state_of_being_nlp):
    text = text or ""
    doc = nlp(text)
    verb_matches = []
    for token in doc:
        if is_state_of_being_token(token):
            verb_matches.append(
                {
                    "text": token.text,
                    "lemma": token.lemma_,
                    "pos": token.pos_,
                    "tag": token.tag_,
                    "start": token.idx,
                    "end": token.idx + len(token.text),
                }
            )
    contraction_matches = [
        {"text": match.group(0), "start": match.start(), "end": match.end()}
        for match in CONTRACTION_PATTERN.finditer(text)
    ]
    violation_count = len(verb_matches) + len(contraction_matches)
    return {
        "state_of_being_count": len(verb_matches),
        "contraction_count": len(contraction_matches),
        "e_prime_violation_count": violation_count,
        "e_prime_retained": violation_count == 0,
        "e_prime_score": 1.0 if violation_count == 0 else 0.0,
        "state_of_being_matches": verb_matches,
        "contraction_matches": contraction_matches,
    }


def add_e_prime_scores(frame, text_column="reply", nlp=state_of_being_nlp):
    scored = frame.copy()
    checks = [check_e_prime_violations(text, nlp=nlp) for text in scored[text_column].fillna("")]
    for key in ["state_of_being_count", "contraction_count", "e_prime_violation_count", "e_prime_retained", "e_prime_score"]:
        scored[key] = [check[key] for check in checks]
    scored["state_of_being_matches"] = [
        ", ".join(match["text"] for match in check["state_of_being_matches"])
        for check in checks
    ]
    scored["contraction_matches"] = [
        ", ".join(match["text"] for match in check["contraction_matches"])
        for check in checks
    ]
    return scored


results = add_e_prime_scores(results)
results[["condition", "turn", "e_prime_score", "e_prime_violation_count", "state_of_being_matches", "contraction_matches", "reply"]].head()


### Compare Constraint Retention


In [ ]:
summary = results.groupby("condition").agg(
    mean_e_prime_retention=("e_prime_score", "mean"),
    violation_rate=("e_prime_retained", lambda values: 1 - values.mean()),
    mean_violations=("e_prime_violation_count", "mean"),
    mean_state_of_being_count=("state_of_being_count", "mean"),
    mean_contraction_count=("contraction_count", "mean"),
    mean_latency_s=("latency_s", "mean"),
    max_prompt_chars=("prompt_chars", "max"),
)
summary


### Inspect Failures


In [ ]:
failures = results[results["e_prime_score"] < 1.0][
    [
        "condition",
        "turn",
        "user_message",
        "e_prime_violation_count",
        "state_of_being_matches",
        "contraction_matches",
        "reply",
    ]
]
failures


### Save Results And Limitations


In [ ]:
out_dir = Path("/content/spotlight_e_prime_results") if IN_COLAB else Path("notebooks/results")
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = out_dir / f"spotlight_e_prime_constraint_retention_{stamp}.csv"
json_path = out_dir / f"spotlight_e_prime_constraint_retention_{stamp}.json"

results.to_csv(csv_path, index=False)
json_path.write_text(
    json.dumps(
        {
            "model": model,
            "alpha": ALPHA,
            "constraint": E_PRIME_CONSTRAINT,
            "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
            "max_user_turns": MAX_USER_TURNS,
            "history_window_messages": HISTORY_WINDOW_MESSAGES,
            "max_model_len": MAX_MODEL_LEN,
            "summary": summary.reset_index().to_dict(orient="records"),
            "limitations": [
                "Notebook smoke validation checks structure and platform-specific imports only unless this notebook is run in Colab.",
                "The contraction checker uses an explicit list and may not catch every informal contraction.",
                "The spaCy POS-based checker can miss malformed output or mis-tag unusual phrasing.",
            ],
            "rows": results.to_dict(orient="records"),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Saved CSV: {csv_path}")
print(f"Saved JSON: {json_path}")
print("Validation note: this committed notebook has been smoke-validated locally; run it in a Colab GPU runtime for execution results.")


### Track Frozen MLR-19 Reference Run In W&B


In [ ]:
# Optional W&B tracking for the frozen MLR-19 reference run.
# This cell does not change the experiment, scoring, or saved local result files.
ENABLE_WANDB = os.environ.get("WANDB_MODE", "").lower() not in {"disabled", "off"}
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "vllm-hook-eprime")
WANDB_GROUP = os.environ.get("WANDB_GROUP", "MLR-20-MLR-19-reference")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", "MLR-19-checkpoint-eprime-alpha-0.20")

# Optional one-time paste for notebook runs. Leave blank to use the environment
# or Colab secret named WANDB_API_KEY. Do not save/share the notebook with a key filled in.
WANDB_API_KEY = ""
MLR19_CHECKPOINT = "f49fc8980656020bdff8911aee2dbd5196961316"

if ENABLE_WANDB:
    if WANDB_API_KEY.strip():
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
    elif IN_COLAB:
        try:
            from google.colab import userdata
            wandb_api_key = userdata.get("WANDB_API_KEY")
            if wandb_api_key:
                os.environ["WANDB_API_KEY"] = wandb_api_key
        except Exception:
            pass

condition = ExperimentCondition(
    condition_id="MLR-19-eprime-constraint-retention",
    comparison_group=WANDB_GROUP,
    spotlight=True,
    alpha=ALPHA,
    implementation="frozen-MLR-19-baseline-and-spotlight-reference",
    model=model,
    constraint_formulation="E-Prime",
    constraint_complexity="state-of-being-verbs-and-listed-contractions",
    intervention_timing="prefill",
    history="rolling-long-conversation",
    turns=MAX_USER_TURNS,
    replicate=1,
    temperature=SAMPLING_PARAMS.temperature,
    max_tokens=SAMPLING_PARAMS.max_tokens,
    history_window_messages=HISTORY_WINDOW_MESSAGES,
    seed=None,
    notebook="notebooks/demo_spotlight_e_prime_colab.ipynb",
    tags=("MLR-19", "MLR-20", "reference", "e-prime", "spotlight"),
    extra={
        "mlr19_checkpoint": MLR19_CHECKPOINT,
        "max_model_len": MAX_MODEL_LEN,
        "baseline_included": True,
        "spotlight_condition_name": "spotlight",
        "baseline_condition_name": "baseline",
    },
)

turn_results = [
    TurnResult(
        condition=row["condition"],
        turn=int(row["turn"]),
        prompt=row["user_message"],
        response=row["reply"],
        compliant=bool(row["e_prime_retained"]),
        violation_count=int(row["e_prime_violation_count"]),
        state_of_being_count=int(row["state_of_being_count"]),
        contraction_count=int(row["contraction_count"]),
        extra={
            "prompt_chars": int(row["prompt_chars"]),
            "history_messages_in_prompt": int(row["history_messages_in_prompt"]),
            "latency_s": float(row["latency_s"]),
            "state_of_being_matches": row["state_of_being_matches"],
            "contraction_matches": row["contraction_matches"],
        },
    )
    for row in results.to_dict(orient="records")
]

spotlight_turns = [turn for turn in turn_results if turn.condition == "spotlight"]
metrics = RunMetrics.from_turns(spotlight_turns)
provenance = collect_runtime_provenance(REPO_ROOT)
provenance.update(
    {
        "mlr19_checkpoint": MLR19_CHECKPOINT,
        "results_csv": str(csv_path),
        "results_json": str(json_path),
    }
)

tracked_result = ExperimentResult(
    condition=condition,
    turns=turn_results,
    metrics=metrics,
    provenance=provenance,
    full_result={
        "model": model,
        "alpha": ALPHA,
        "constraint": E_PRIME_CONSTRAINT,
        "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
        "max_user_turns": MAX_USER_TURNS,
        "history_window_messages": HISTORY_WINDOW_MESSAGES,
        "max_model_len": MAX_MODEL_LEN,
        "summary": summary.reset_index().to_dict(orient="records"),
        "rows": results.to_dict(orient="records"),
        "limitations": [
            "Notebook smoke validation checks structure and platform-specific imports only unless this notebook is run in Colab.",
            "The contraction checker uses an explicit list and may not catch every informal contraction.",
            "The spaCy POS-based checker can miss malformed output or mis-tag unusual phrasing.",
        ],
    },
)

tracker = WandbTracker(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    run_name=WANDB_RUN_NAME,
    tags=list(condition.tags),
    enabled=ENABLE_WANDB,
)
wandb_payload = tracker.log_result(tracked_result)

if ENABLE_WANDB:
    print(f"Logged W&B run: project={WANDB_PROJECT} group={WANDB_GROUP} run={WANDB_RUN_NAME}")
else:
    print("W&B disabled; built tracking payload without logging.")

print("Tracked metrics:", wandb_payload["metrics"])
